In [1]:
#Cell 1 — Imports and Load Everything
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import anthropic
import json
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
sns.set_theme(style='whitegrid')

# Install anthropic SDK if needed
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'anthropic'], check=True)

# Reload after install
import anthropic

# Load all outputs from previous days
shap_df = pd.read_parquet('../data/features/shap_values.parquet')
cf_df   = pd.read_parquet('../data/features/counterfactuals.parquet')
shap_df['date'] = pd.to_datetime(shap_df['date'])

FEATURE_COLS = [
    'lag_7', 'lag_14', 'lag_28',
    'roll_mean_7', 'roll_mean_14', 'roll_mean_28',
    'roll_std_7', 'roll_std_14', 'roll_std_28',
    'day_of_week', 'day_of_month', 'week_of_year',
    'quarter', 'is_weekend', 'is_month_start', 'is_month_end',
    'is_snap', 'is_event', 'is_sporting_event',
    'is_cultural_event', 'is_national_holiday', 'is_religious_event',
    'sell_price', 'price_change_pct', 'price_vs_mean', 'is_price_drop',
]

print("✅ All data loaded")
print(f"   SHAP data : {shap_df.shape}")
print(f"   CF data   : {cf_df.shape}")

✅ All data loaded
   SHAP data : (1400, 58)
   CF data   : (6, 8)


In [ ]:
#Cell 2 - Set Up Anthropic API
import sys, subprocess
import os 
subprocess.run([sys.executable, '-m', 'pip', 'install', 'groq'], check=True)

from groq import Groq

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Quick test
test = client.chat.completions.create(
    model      = "llama-3.3-70b-versatile",
    messages   = [{"role": "user", "content": "Say: Groq API connection successful"}],
    max_tokens = 20
)
print(f"✅ {test.choices[0].message.content}")

✅ Groq API connection successful.


In [10]:
#Cell 3— Build the Report Data Package
import math
def build_report_data(product_id, shap_df, cf_df, FEATURE_COLS):
    """
    Assembles all model outputs for one product into a
    structured dict ready to send to the LLM.
    """
    product_data = shap_df[shap_df['item_id'] == product_id].copy()

    if len(product_data) == 0:
        raise ValueError(f"Product {product_id} not found in SHAP data")

    # Get SHAP feature importances for this product
    shap_cols = [f'shap_{c}' for c in FEATURE_COLS]
    mean_shap = product_data[shap_cols].abs().mean()
    mean_shap.index = [c.replace('shap_', '') for c in mean_shap.index]
    top_features = mean_shap.sort_values(ascending=False).head(5)

    # Get latest prediction
    latest = product_data.sort_values('date').iloc[-1]

    # Get counterfactual scenarios
    cf_scenarios = cf_df[['scenario', 'base_pred',
                           'cf_pred', 'delta', 'delta_pct']].to_dict('records')

    # Sales trend — last 7 days avg vs last 28 days avg
    trend_7  = float(latest['roll_mean_7'])
    trend_28 = float(latest['roll_mean_28'])
    trend_direction = "upward" if trend_7 > trend_28 else "downward"

    report_data = {
        "product_id"        : product_id,
        "store"             : "CA_1",
        "date"              : str(latest['date'].date()),
        "reorder_qty"       : max(1, math.ceil(float(latest['predicted']) * 1.3)),
        "actual_sold"       : round(float(latest['units_sold']), 1),
        "sell_price"        : round(float(latest['sell_price']), 2),
        "is_snap_day"       : bool(latest['is_snap']),
        "is_event_day"      : bool(latest['is_event']),
        "trend_7day_avg"    : round(trend_7, 2),
        "trend_28day_avg"   : round(trend_28, 2),
        "trend_direction"   : trend_direction,
        "top_features"      : {k: round(float(v), 4)
                                for k, v in top_features.items()},
        "counterfactuals"   : cf_scenarios,
        "snap_demand_lift_pct" : 25.5,
        "price_elasticity"  : -0.445,
    }

    return report_data

# Test it
product_id  = 'FOODS_3_561'
report_data = build_report_data(product_id, shap_df, cf_df, FEATURE_COLS)

print("✅ Report data package built:")
print(json.dumps(report_data, indent=2))

✅ Report data package built:
{
  "product_id": "FOODS_3_561",
  "store": "CA_1",
  "date": "2016-04-24",
  "reorder_qty": 11,
  "actual_sold": 4.0,
  "sell_price": 2.98,
  "is_snap_day": false,
  "is_event_day": false,
  "trend_7day_avg": 5.0,
  "trend_28day_avg": 7.71,
  "trend_direction": "downward",
  "top_features": {
    "roll_mean_7": 1.2925,
    "roll_mean_28": 1.0582,
    "roll_mean_14": 0.9053,
    "lag_7": 0.6334,
    "day_of_week": 0.4711
  },
  "counterfactuals": [
    {
      "scenario": "Remove SNAP day",
      "base_pred": 7.441153526306152,
      "cf_pred": 7.349318981170654,
      "delta": -0.09183454513549805,
      "delta_pct": -1.2341439359828805
    },
    {
      "scenario": "Price +20%",
      "base_pred": 7.441153526306152,
      "cf_pred": 7.441153526306152,
      "delta": 0.0,
      "delta_pct": 0.0
    },
    {
      "scenario": "Price -20%",
      "base_pred": 7.441153526306152,
      "cf_pred": 7.632298946380615,
      "delta": 0.1911454200744629,
      "de

In [11]:
#Cell 4 - Generate Seller Report with LLM
def generate_seller_report(report_data, client):
    system_prompt = """You are ExplainStock, an AI system built to explain 
automated inventory decisions to Amazon third-party sellers in plain English.

Your job is to take structured model output — predictions, SHAP feature 
importances, and counterfactual scenarios — and write a clear, helpful 
report that a seller with no data science background can understand and 
act on.

Rules:
- Write for a non-technical seller, not a data scientist
- Always explain WHY the system made its recommendation
- Always include 2-3 specific actions the seller can take
- Be concise — maximum 300 words
- Use simple language, no jargon
- Format with clear sections: Summary, Why This Recommendation, 
  What Could Change It, Your Action Items
- Use specific numbers from the data, never be vague"""

    user_prompt = f"""Generate an inventory decision report for this seller.

Here is the model output data:
{json.dumps(report_data, indent=2)}

Additional context:
- roll_mean_7 = average daily sales over last 7 days
- roll_mean_28 = average daily sales over last 28 days
- SHAP values measure how much each factor pushed the forecast up or down
- Counterfactuals show what the recommendation would be under different conditions
- SNAP days are government benefit days that increase FOODS demand by ~25%

Write the seller report now."""

    response = client.chat.completions.create(
        model    = "llama-3.3-70b-versatile",
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        max_tokens  = 600,
        temperature = 0.3
    )

    return response.choices[0].message.content

# Generate the report
print("Generating seller report...")
report = generate_seller_report(report_data, client)

print("\n" + "=" * 60)
print("EXPLAINSTOCK — AI-GENERATED SELLER REPORT")
print("=" * 60)
print(report)
print("=" * 60)

Generating seller report...

EXPLAINSTOCK — AI-GENERATED SELLER REPORT
## Summary
We recommend reordering 11 units of product FOODS_3_561 for your CA_1 store. This is based on a predicted demand of 7.44 units, considering the current downward trend in sales.

## Why This Recommendation
The model considered several factors, including the average daily sales over the last 7 and 28 days (5.0 and 7.71 units, respectively). The top factors influencing this prediction are the 7-day and 28-day average sales, with SHAP values of 1.29 and 1.06, respectively. This means that the recent sales trend has a significant impact on our recommendation.

## What Could Change It
If the demand trend increases by 50%, our recommendation would increase to 11.03 units. On the other hand, if the demand trend decreases by 40%, our recommendation would decrease to 6.10 units. Additionally, if the price is decreased by 20%, our recommendation would increase to 7.64 units, indicating that customers are sensitive t

In [12]:
#Cell 5 — Batch Reports for Multiple Products
def generate_batch_reports(shap_df, cf_df, client, FEATURE_COLS, n=5):
    """Generate reports for n different products."""

    # Pick products with varied characteristics
    products = shap_df.groupby('item_id').agg(
        avg_pred  = ('predicted', 'mean'),
        avg_sales = ('units_sold', 'mean')
    ).reset_index()

    # Sample across different demand levels
    products = products.sort_values('avg_pred')
    indices  = np.linspace(0, len(products)-1, n, dtype=int)
    selected = products.iloc[indices]['item_id'].tolist()

    reports = {}
    for pid in selected:
        try:
            data   = build_report_data(pid, shap_df, cf_df, FEATURE_COLS)
            report = generate_seller_report(data, client)
            reports[pid] = {
                'data'  : data,
                'report': report
            }
            print(f"✅ Generated report for {pid} "
                  f"(avg demand: {data['reorder_qty']:.1f} units)")
        except Exception as e:
            print(f"⚠️  Skipped {pid}: {e}")

    return reports

print("Generating batch reports for 5 products...")
batch_reports = generate_batch_reports(
    shap_df, cf_df, client, FEATURE_COLS, n=5
)

print(f"\n✅ Generated {len(batch_reports)} reports")

Generating batch reports for 5 products...
✅ Generated report for FOODS_1_072 (avg demand: 1.0 units)
✅ Generated report for FOODS_3_296 (avg demand: 1.0 units)
✅ Generated report for FOODS_3_567 (avg demand: 2.0 units)
✅ Generated report for FOODS_3_211 (avg demand: 4.0 units)
✅ Generated report for FOODS_3_288 (avg demand: 25.0 units)

✅ Generated 5 reports


In [13]:
import os
import re
from datetime import datetime

# Clean markdown symbols from report text
def clean_report(text):
    text = re.sub(r'##\s*', '', text)           # remove ## headers
    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text) # remove **bold**
    return text.strip()

# Save as JSON
with open('../outputs/reports/seller_reports.json', 'w') as f:
    serializable = {}
    for pid, content in batch_reports.items():
        serializable[pid] = {
            'data'  : content['data'],
            'report': content['report']
        }
    json.dump(serializable, f, indent=2, default=str)

print("✅ Saved: outputs/reports/seller_reports.json")

# Create HTML report
html_template = """
<!DOCTYPE html>
<html>
<head>
    <title>ExplainStock — Seller Reports</title>
    <style>
        body {{ font-family: Arial, sans-serif; max-width: 900px;
                margin: 40px auto; padding: 20px; background: #f5f5f5; }}
        .report-card {{ background: white; border-radius: 8px;
                        padding: 24px; margin-bottom: 24px;
                        box-shadow: 0 2px 8px rgba(0,0,0,0.1); }}
        .product-header {{ display: flex; justify-content: space-between;
                           align-items: center; margin-bottom: 16px; }}
        .product-id {{ font-size: 20px; font-weight: bold; color: #1565C0; }}
        .reorder-badge {{ background: #1B5E20; color: white;
                          padding: 8px 16px; border-radius: 20px;
                          font-size: 18px; font-weight: bold; }}
        .meta {{ color: #666; font-size: 13px; margin-bottom: 16px; }}
        .snap-badge {{ background: #E3F2FD; color: #1565C0;
                       padding: 3px 10px; border-radius: 12px;
                       font-size: 12px; margin-left: 8px; }}
        .report-text {{ line-height: 1.8; color: #333; white-space: pre-wrap; }}
        .section-title {{ font-weight: bold; color: #1565C0;
                          margin-top: 12px; margin-bottom: 4px; }}
        h1 {{ color: #1565C0; border-bottom: 3px solid #1565C0;
              padding-bottom: 12px; }}
        .footer {{ text-align: center; color: #999; margin-top: 40px;
                   font-size: 12px; }}
    </style>
</head>
<body>
    <h1>&#127978; ExplainStock — Inventory Decision Reports</h1>
    <p style="color:#666">Store: CA_1 | Generated: {date} |
       Powered by XGBoost + SHAP + LLM Narratives</p>
    {cards}
    <div class="footer">
        ExplainStock — Explainable Inventory Decisions for Amazon Sellers<br>
        Built with XGBoost &middot; SHAP &middot; DiCE Counterfactuals &middot; LLaMA 3.3
    </div>
</body>
</html>
"""

cards_html = ""
for pid, content in batch_reports.items():
    d           = content['data']
    snap        = '<span class="snap-badge">&#128197; SNAP Day</span>' \
                  if d['is_snap_day'] else ''
    clean_text  = clean_report(content['report'])

    cards_html += f"""
    <div class="report-card">
        <div class="product-header">
            <div>
                <div class="product-id">{pid} {snap}</div>
                <div class="meta">
                    Store: {d['store']} &nbsp;|&nbsp;
                    Date: {d['date']} &nbsp;|&nbsp;
                    Price: ${d['sell_price']} &nbsp;|&nbsp;
                    7-day avg: {d['trend_7day_avg']} units/day
                </div>
            </div>
            <div class="reorder-badge">
                Reorder: {d['reorder_qty']} units
            </div>
        </div>
        <div class="report-text">{clean_text}</div>
    </div>
    """

html_output = html_template.format(
    date  = datetime.now().strftime("%Y-%m-%d %H:%M"),
    cards = cards_html
)

with open('../outputs/reports/explainstock_report.html', 'w',
          encoding='utf-8') as f:
    f.write(html_output)

print("✅ Saved: outputs/reports/explainstock_report.html")
print("   Open this file in your browser to see the full demo")

✅ Saved: outputs/reports/seller_reports.json
✅ Saved: outputs/reports/explainstock_report.html
   Open this file in your browser to see the full demo


In [14]:
#Cell 7 — Final Project Summary
print("=" * 60)
print("EXPLAINSTOCK — COMPLETE PROJECT SUMMARY")
print("Month 1 Complete")
print("=" * 60)
print(f"""
WHAT YOU BUILT
  An end-to-end Explainable Inventory Decision System
  inspired by Amazon SCOT's seller transparency problem.

TECHNICAL STACK
  • XGBoost demand forecasting (Test RMSE: 2.49, MAE: 1.35)
  • 27 engineered features across lag, rolling, calendar, price
  • SHAP global + local explainability
  • Counterfactual scenario analysis (custom DiCE-style engine)
  • LLM narrative report generation (Claude API)

KEY FINDINGS
  • 7-day rolling demand is the dominant forecast signal
  • SNAP days drive +25.5% demand lift in FOODS category
  • Price elasticity: -0.445 (moderate inelasticity)
  • Demand trend changes drive 18-48% reorder adjustments
  • Weekday vs weekend accounts for ~9% demand difference

DATASET
  • M5 Forecasting Competition (Walmart US)
  • 2,708,745 rows after feature engineering
  • CA_1 store, FOODS category, 1,437 products
  • 5+ years of daily sales history

OUTPUT FILES
  models/   xgb_ca1_foods_v1.json
  features/ ca1_foods_features.parquet
            shap_values.parquet
            counterfactuals.parquet
  reports/  seller_reports.json
            explainstock_report.html
  plots/    07a through 12 (6 research-grade plots)

NEXT STEPS (Month 2)
  • LightGBM comparison model
  • DiCE formal counterfactuals (once Pandas compat fixed)
  • India festival calendar extension
  • Streamlit dashboard for live demo
  • GitHub README and project write-up
""")
print("=" * 60)
print("✅ Day 9 complete. Month 1 complete.")

EXPLAINSTOCK — COMPLETE PROJECT SUMMARY
Month 1 Complete

WHAT YOU BUILT
  An end-to-end Explainable Inventory Decision System
  inspired by Amazon SCOT's seller transparency problem.

TECHNICAL STACK
  • XGBoost demand forecasting (Test RMSE: 2.49, MAE: 1.35)
  • 27 engineered features across lag, rolling, calendar, price
  • SHAP global + local explainability
  • Counterfactual scenario analysis (custom DiCE-style engine)
  • LLM narrative report generation (Claude API)

KEY FINDINGS
  • 7-day rolling demand is the dominant forecast signal
  • SNAP days drive +25.5% demand lift in FOODS category
  • Price elasticity: -0.445 (moderate inelasticity)
  • Demand trend changes drive 18-48% reorder adjustments
  • Weekday vs weekend accounts for ~9% demand difference

DATASET
  • M5 Forecasting Competition (Walmart US)
  • 2,708,745 rows after feature engineering
  • CA_1 store, FOODS category, 1,437 products
  • 5+ years of daily sales history

OUTPUT FILES
  models/   xgb_ca1_foods_v1.js